# 22DM015 Final Project — Financial PhraseBank
## Part 3d — methodology analysis

This notebook is the student-authored Part 3d write-up.‍ It reads the shared scoreboard `results/results.csv` (produced by `part1.ipynb`, `bert_part2.ipynb`, `zeroshot_part2.ipynb` and `part3.ipynb`) and expresses every Part 2 technique as a point on the Part 3 data-scaling curve, so the methods can be compared on one common scale.‍ No model is trained here.‍

In [ ]:
# watermark: AGLLM (AI-assisted content disclosure)
# Part 3d: place each Part 2 technique on the Part 3 data-scaling curve as a "real-label
# equivalent" — how many real labels the curve needs to match its macro-F1. To keep the
# comparison apples-to-apples, the equivalent is computed from each technique's row trained
# under the SAME protocol as the curve (the '-curveproto' rows from part3.ipynb); the Part 2
# deliverable score (protocol 128/8/20) is shown alongside for context.
# No training/API here — reads results/results.csv only (run part3.ipynb's Part 3c cell to
# populate the '-curveproto' rows; the zero-shot row comes from zeroshot_part2.ipynb).
import sys
import numpy as np
import pandas as pd
sys.path.append('..')
import eval_utils as eu

MODEL = 'bert-base-uncased'
res = pd.read_csv(eu.RESULTS_CSV)
res = res[res['split'] == 'test']

# The data-scaling curve (plain BERT, protocol 64/16/3) — the yardstick.
curve = (res[(res['model'] == MODEL) & res['method'].str.startswith('full-')]
         .sort_values('n_train_labeled'))


def row_of(model, method):
    hit = res[(res['model'] == model) & (res['method'] == method)]
    if not len(hit):
        return None
    r = hit.iloc[-1]
    return {'f1': float(r['f1_macro']), 'n': int(r['n_train_labeled'])}


def eq_str(f1):
    """Real labels the curve needs to reach macro-F1 `f1`, by linear interpolation on the
    curve. The curve's f1 is replaced by its running max (best F1 achievable with <= n
    labels) so a dip at the top — e.g. full-100% below full-75% — can't break the lookup.
    '>full data' if above the curve's best, '<n_min' if below its smallest point."""
    if f1 is None:
        return None
    xs = curve['n_train_labeled'].to_numpy(float)
    ys = np.maximum.accumulate(curve['f1_macro'].to_numpy(float))   # best F1 with <= n labels
    if f1 > ys.max():
        return '>full data'
    if f1 < ys.min():
        return f'<{int(xs.min())}'
    return round(float(np.interp(f1, ys, xs)))


# (label, model, base method). The curve-protocol row is base+'-curveproto'; the zero-shot
# LLM has no training protocol, so its single score is used directly.
TECHNIQUES = [('2a 32-shot',          'bert-base-uncased', '32-shot'),
              ('2b back-translation', 'bert-base-uncased', 'augmented'),
              ('2d LLM-generated',    'bert-base-uncased', 'llm-generated'),
              ('2e optimal-combo',    'bert-base-uncased', 'optimal-combo')]

rows = []
for label, model, method in TECHNIQUES:
    p2 = row_of(model, method)                       # Part 2 deliverable protocol (128/8/20)
    cp = row_of(model, f'{method}-curveproto')        # curve protocol (64/16/3) = apples-to-apples
    if cp is None:
        print(f'[waiting] curve-protocol row not logged yet for: {label} '
              f"(run part3.ipynb's Part 3c cell)")
    rows.append({'technique': label,
                 'rows_used': (p2 or cp or {}).get('n'),
                 'f1_part2_128/8/20': round(p2['f1'], 4) if p2 else None,
                 'f1_curve_64/16/3': round(cp['f1'], 4) if cp else None,
                 'real_label_equiv (exact)': eq_str(cp['f1']) if cp else None})

# zero-shot LLM: protocol-independent reference (no training), shown on its own line.
zs = row_of('claude-opus-4-8', 'zero-shot')
if zs:
    rows.append({'technique': '2c zero-shot (LLM)', 'rows_used': 0,
                 'f1_part2_128/8/20': round(zs['f1'], 4), 'f1_curve_64/16/3': None,
                 'real_label_equiv (exact)': eq_str(zs['f1'])})

print('Data-scaling curve (plain BERT, protocol max_len 64 / batch 16 / 3 epochs):')
print(curve[['method', 'n_train_labeled', 'f1_macro']].to_string(index=False), '\n')
print('Each Part 2 technique: its Part 2 deliverable score, its score re-trained on the '
      'curve protocol, and the exact real-label equivalent read off the curve from the '
      'curve-protocol score:')
summary = pd.DataFrame(rows)
summary

### 3d - methodology analysis

Part 3 builds a data-scaling curve by training the same `bert-base-uncased` on 1, 10, 25, 50, 75 and 100 percent of the training set — train and val folded together, so the full-data point is 1,811 examples — and scoring every point on the shared `test` split.‍ The curve is the yardstick: each Part 2 technique is placed on it at the macro-F1 it reaches, and we read off how many real labels the curve needs to match that score.‍ To make this an exact comparison rather than an indicative one, every Part 2 technique was re-trained under the same protocol as the curve (`max_len 64`, batch 16, 3 epochs); these are the `-curveproto` rows, and the real-label equivalents below come from them.‍ The Part 2 deliverable scores (protocol `max_len 128`, batch 8, 20 epochs) are kept in the table only for context.‍

Under the shared protocol the ranking is clear.‍ The zero-shot LLM is off the chart at 0.978, above the curve's best of about 0.968 at full data, so no amount of real data fed to BERT matches it.‍ Among the limited-label BERT techniques, the pooled optimal-combo is worth about 774 real labels (0.830), LLM-generated about 726 (0.789), back-translation about 515 (0.609), and the plain 32-shot model collapses to 0.237, below even the smallest curve point, so it is worth fewer than 18 real labels.‍ The two transparent baselines sit far lower: the majority-class and random floors are around 0.25 to 0.33, and the rule-based lexicon reaches 0.788, which the curve only overtakes somewhere between 25 and 50 percent of the data.‍ The reading is that on this light protocol training volume dominates: more rows, even noisy ones, help, because three epochs badly under-train the small sets, which is also why 32-shot on its own barely learns.‍

The comparison does more than remove the protocol caveat, it overturns one of the Part 2 conclusions.‍ At the Part 2 protocol (20 epochs) LLM-generated alone (0.841) beat the pooled optimal-combo (0.800), and Part 2e read this as curation beating blind concatenation.‍ Under the curve's 3-epoch protocol the order reverses: optimal-combo (0.830) now beats LLM-generated (0.789).‍ The back-translation paraphrases that looked like harmful noise at 20 epochs behave like useful extra volume at 3 epochs, when the model is data-starved rather than at risk of fitting the noise.‍ So the Part 2e lesson is real but protocol-specific: curation wins when the model is trained long enough to overfit low-quality data, and volume wins when it is under-trained.‍ This is exactly what the apples-to-apples comparison is for.‍

Three caveats remain.‍ First, three epochs under-trains the small sets, most visibly in the 32-shot collapse; a comparison at more epochs would lift the limited-label points but cost the curve much more compute.‍ Second, the curve is noisy at the low end and even dips at the top — full-100% at 0.963 sits below full-75% at 0.968 — and every point is a single-seed estimate, so the crossing points are approximate (the real-label equivalent uses the curve's running-best F1 to stay well defined).‍ Third, the zero-shot row carries the Part 2c caveats: Financial PhraseBank is a public 2014 dataset a frontier model has likely seen, so its 0.978 may partly reflect memorisation, and it is a paid black-box API set against deterministic offline BERT.‍

With more compute the next steps are to repeat the whole comparison over several seeds and at a heavier shared protocol so the equivalents are both exact and stable, to add a domain state-of-the-art model such as `ProsusAI/finbert` as a fine-tuned ceiling for the curve, and to test the contamination question for the LLM on an out-of-distribution split postdating the model's training cut-off.‍